<a href="https://colab.research.google.com/github/Nadsyuhamus/Databladez_Tourism/blob/feature%2Fml-tourism-engine/04_district_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%cd /content/Databladez_Tourism

!git branch --show-current
!git pull origin feature/ml-tourism-engine

!ls -lh ml/outputs

[Errno 2] No such file or directory: '/content/Databladez_Tourism'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
ls: cannot access 'ml/outputs': No such file or directory


In [6]:
from pathlib import Path

matches = list(
    Path("/content").rglob("demand_forecast_latest.csv")
)

print("Matches found:", len(matches))

for path in matches:
    print(path)

Matches found: 0


In [7]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

REPO_DIR = Path("/content/Databladez_Tourism")
DATA_DIR = REPO_DIR / "outputs" / "datathon_cleaned_all_files"
ML_OUTPUT_DIR = REPO_DIR / "ml" / "outputs"

forecast_path = ML_OUTPUT_DIR / "demand_forecast_latest.csv"
context_path = DATA_DIR / "district_context.csv"
mapping_path = DATA_DIR / "district_name_mapping.csv"

for path in [forecast_path, context_path, mapping_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

forecast = pd.read_csv(forecast_path)
district_context = pd.read_csv(context_path)
name_mapping = pd.read_csv(mapping_path)

required_forecast_columns = {
    "series_id",
    "state",
    "area",
    "geo_level",
}

missing_columns = required_forecast_columns - set(forecast.columns)

if missing_columns:
    raise ValueError(
        f"demand_forecast_latest.csv is missing: {sorted(missing_columns)}"
    )

district_forecast = (
    forecast.loc[
        forecast["geo_level"].eq("district_or_area"),
        ["series_id", "state", "area", "geo_level"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("District/area forecast series:", len(district_forecast))
print("District context rows:", len(district_context))
print("District-name mapping rows:", len(name_mapping))

FileNotFoundError: Missing required file: /content/Databladez_Tourism/ml/outputs/demand_forecast_latest.csv

In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode()
    value = value.upper()
    value = re.sub(r"[^A-Z0-9]+", " ", value)

    return value.strip()


# Prepare the supplied name mapping.
name_mapping["source_key"] = (
    name_mapping["source_district_or_area_name"]
    .map(normalize_name)
)

# Confirm that repeated spelling variants do not point to
# different canonical districts.
mapping_conflicts = (
    name_mapping.groupby("source_key")["canonical_district"]
    .nunique()
)

mapping_conflicts = mapping_conflicts[mapping_conflicts > 1]

if not mapping_conflicts.empty:
    raise ValueError(
        "Conflicting canonical mappings found:\n"
        f"{mapping_conflicts}"
    )

mapping_unique = (
    name_mapping[
        [
            "source_key",
            "canonical_district",
            "needs_review",
        ]
    ]
    .drop_duplicates("source_key")
)

district_forecast["source_key"] = (
    district_forecast["area"].map(normalize_name)
)

crosswalk = district_forecast.merge(
    mapping_unique,
    on="source_key",
    how="left",
    validate="many_to_one",
)

crosswalk["mapping_method"] = "supplied_mapping"
crosswalk["mapping_confidence"] = "high"
crosswalk["mapping_note"] = "Mapped using district_name_mapping.csv"


# Explicitly reviewed differences between the Google Trends
# labels and DOSM district-context labels.
manual_overrides = pd.DataFrame(
    [
        {
            "state": "Selangor",
            "area": "Hulu Langat",
            "canonical_override": "Ulu Langat",
            "override_confidence": "high",
            "override_note": (
                "Spelling variant: Google Trends uses Hulu; "
                "DOSM context uses Ulu."
            ),
        },
        {
            "state": "Selangor",
            "area": "Hulu Selangor",
            "canonical_override": "Ulu Selangor",
            "override_confidence": "high",
            "override_note": (
                "Spelling variant: Google Trends uses Hulu; "
                "DOSM context uses Ulu."
            ),
        },
        {
            "state": "Sarawak",
            "area": "Tanjung",
            "canonical_override": "Tanjung Manis",
            "override_confidence": "medium",
            "override_note": (
                "Shortened Google Trends label matched to the "
                "corresponding DOSM context district in Sarawak."
            ),
        },
    ]
)

crosswalk = crosswalk.merge(
    manual_overrides,
    on=["state", "area"],
    how="left",
    validate="one_to_one",
)

override_mask = crosswalk["canonical_override"].notna()

crosswalk.loc[override_mask, "canonical_district"] = (
    crosswalk.loc[override_mask, "canonical_override"]
)

crosswalk.loc[override_mask, "mapping_method"] = (
    "manual_reviewed_override"
)

crosswalk.loc[override_mask, "mapping_confidence"] = (
    crosswalk.loc[override_mask, "override_confidence"]
)

crosswalk.loc[override_mask, "mapping_note"] = (
    crosswalk.loc[override_mask, "override_note"]
)

In [ ]:
context_keys = (
    district_context[["state", "district"]]
    .drop_duplicates()
    .copy()
)

context_keys["state_key"] = (
    context_keys["state"].map(normalize_name)
)

context_keys["district_key"] = (
    context_keys["district"].map(normalize_name)
)

crosswalk["state_key"] = crosswalk["state"].map(normalize_name)

crosswalk["district_key"] = (
    crosswalk["canonical_district"].map(normalize_name)
)

crosswalk = crosswalk.merge(
    context_keys[
        [
            "state_key",
            "district_key",
            "district",
        ]
    ].rename(columns={"district": "matched_context_district"}),
    on=["state_key", "district_key"],
    how="left",
    validate="many_to_one",
)

crosswalk["context_match"] = (
    crosswalk["matched_context_district"].notna()
)

unmatched = crosswalk.loc[
    ~crosswalk["context_match"],
    [
        "series_id",
        "state",
        "area",
        "canonical_district",
        "mapping_method",
    ],
]

print("Forecast district/area series:", len(crosswalk))
print("Matched to DOSM context:", int(crosswalk["context_match"].sum()))
print("Unmatched:", len(unmatched))
print(
    "Manual reviewed overrides:",
    int(crosswalk["mapping_method"].eq(
        "manual_reviewed_override"
    ).sum()),
)

if not unmatched.empty:
    display(unmatched)
    raise ValueError(
        "Some forecast areas remain unmatched. "
        "Review them before continuing."
    )

assert crosswalk["series_id"].is_unique
assert len(crosswalk) == len(district_forecast)

print("\n✓ Every forecast area has exactly one district-context match.")